In [ ]:
# =========================================================
# 0. Imports
# =========================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from category_encoders import TargetEncoder

In [ ]:
# =========================================================
# 1. Utility functions
# =========================================================

def safe_divide(numerator, denominator, fill_value=0):
    """
    분모가 0이면 fill_value 반환.
    champion pipeline에서는 ratio 계산 불가능한 경우 0으로 처리.
    """
    numerator = pd.to_numeric(numerator, errors="coerce").fillna(0)
    denominator = pd.to_numeric(denominator, errors="coerce").fillna(0)

    return np.where(
        denominator == 0,
        fill_value,
        numerator / denominator
    )


def map_count_value(x):
    """
    '0회', '1회', ..., '6회 이상' 형태의 횟수형 문자열을 숫자로 변환.
    이미 숫자인 경우도 최대한 유지.
    """
    if pd.isna(x):
        return 0

    count_map = {
        "0회": 0,
        "1회": 1,
        "2회": 2,
        "3회": 3,
        "4회": 4,
        "5회": 5,
        "6회 이상": 6,
    }

    if x in count_map:
        return count_map[x]

    try:
        return float(x)
    except Exception:
        return 0

In [ ]:
# =========================================================
# 2. Main preprocessing + feature engineering
# =========================================================

def data_preprocessing(df):
    """
    Champion 전처리/FE 정리본.

    핵심 아이디어:
    1. 경과일 결측 여부를 performed flag로 보존
    2. 횟수형 문자열을 숫자로 변환
    3. 불임 원인 요약 feature 생성
    4. 배아/난자 효율 ratio 생성
    5. 고령 여부 및 고령 interaction 생성
    6. 범주형 컬럼은 문자열로 통일
    """
    df = df.copy()

    # -----------------------------------------------------
    # 1) 경과일 performed flag
    # -----------------------------------------------------
    time_cols = [
        "임신 시도 또는 마지막 임신 경과 연수",
        "난자 해동 경과일",
        "난자 혼합 경과일",
        "배아 이식 경과일",
        "배아 해동 경과일",
    ]

    for col in time_cols:
        if col in df.columns:
            df[f"{col}_performed"] = df[col].notna().astype(int)

    # -----------------------------------------------------
    # 2) 횟수형 문자열 숫자화
    # -----------------------------------------------------
    count_cols = [
        "총 시술 횟수",
        "클리닉 내 총 시술 횟수",
        "IVF 시술 횟수",
        "DI 시술 횟수",
        "총 임신 횟수",
        "IVF 임신 횟수",
        "DI 임신 횟수",
        "총 출산 횟수",
        "IVF 출산 횟수",
        "DI 출산 횟수",
    ]

    for col in count_cols:
        if col in df.columns:
            df[col] = df[col].apply(map_count_value).astype(float)

    # -----------------------------------------------------
    # 3) 결측 처리
    #    - 숫자형: 0
    #    - 문자/범주형: Unknown
    # -----------------------------------------------------
    object_cols = df.select_dtypes(include=["object", "category", "string"]).columns

    for col in object_cols:
        df[col] = df[col].astype("object").fillna("Unknown")

    numeric_cols = df.select_dtypes(include=np.number).columns

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # -----------------------------------------------------
    # 4) 불임 원인 요약 feature
    # -----------------------------------------------------
    infertility_cols = [
        "남성 주 불임 원인",
        "남성 부 불임 원인",
        "여성 주 불임 원인",
        "여성 부 불임 원인",
        "부부 주 불임 원인",
        "부부 부 불임 원인",
        "불명확 불임 원인",
        "불임 원인 - 난관 질환",
        "불임 원인 - 남성 요인",
        "불임 원인 - 배란 장애",
        "불임 원인 - 여성 요인",
        "불임 원인 - 자궁경부 문제",
        "불임 원인 - 자궁내막증",
        "불임 원인 - 정자 농도",
        "불임 원인 - 정자 면역학적 요인",
        "불임 원인 - 정자 운동성",
        "불임 원인 - 정자 형태",
    ]

    infertility_cols = [col for col in infertility_cols if col in df.columns]

    male_cols = [
        col for col in infertility_cols
        if ("남성" in col) or ("정자" in col)
    ]

    female_cols = [
        col for col in infertility_cols
        if any(keyword in col for keyword in ["여성", "난관", "배란", "자궁", "내막"])
    ]

    if infertility_cols:
        for col in infertility_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

        df["불임원인_총개수"] = df[infertility_cols].sum(axis=1)

        if male_cols:
            df["남성_원인_수"] = df[male_cols].sum(axis=1)
        else:
            df["남성_원인_수"] = 0

        if female_cols:
            df["여성_원인_수"] = df[female_cols].sum(axis=1)
        else:
            df["여성_원인_수"] = 0

        df["남녀_복합_원인"] = (
            (df["남성_원인_수"] > 0) &
            (df["여성_원인_수"] > 0)
        ).astype(int)

        df["원인불명"] = (
            df["불임원인_총개수"] == 0
        ).astype(int)

    # -----------------------------------------------------
    # 5) 과거 시술/임신/출산 이력 ratio
    # -----------------------------------------------------
    if {"IVF 임신 횟수", "IVF 시술 횟수"}.issubset(df.columns):
        df["IVF_임신성공률"] = safe_divide(
            df["IVF 임신 횟수"],
            df["IVF 시술 횟수"]
        )

    if {"DI 임신 횟수", "DI 시술 횟수"}.issubset(df.columns):
        df["DI_임신성공률"] = safe_divide(
            df["DI 임신 횟수"],
            df["DI 시술 횟수"]
        )

    if {"총 출산 횟수", "총 임신 횟수"}.issubset(df.columns):
        df["출산_임신_전환율"] = safe_divide(
            df["총 출산 횟수"],
            df["총 임신 횟수"]
        )

    if {"클리닉 내 총 시술 횟수", "총 시술 횟수"}.issubset(df.columns):
        df["클리닉_집중도"] = safe_divide(
            df["클리닉 내 총 시술 횟수"],
            df["총 시술 횟수"]
        )

    # -----------------------------------------------------
    # 6) 배아/난자 효율 feature
    # -----------------------------------------------------
    if {"총 생성 배아 수", "혼합된 난자 수"}.issubset(df.columns):
        df["배아_생성률"] = safe_divide(
            df["총 생성 배아 수"],
            df["혼합된 난자 수"]
        )

    if {"이식된 배아 수", "총 생성 배아 수"}.issubset(df.columns):
        df["배아_이식률"] = safe_divide(
            df["이식된 배아 수"],
            df["총 생성 배아 수"]
        )

    if {"저장된 배아 수", "총 생성 배아 수"}.issubset(df.columns):
        df["배아_냉동률"] = safe_divide(
            df["저장된 배아 수"],
            df["총 생성 배아 수"]
        )

    if {"이식된 배아 수", "저장된 배아 수"}.issubset(df.columns):
        denominator = df["이식된 배아 수"] + df["저장된 배아 수"]

        df["배아_이식_집중도"] = safe_divide(
            df["이식된 배아 수"],
            denominator
        )

    # -----------------------------------------------------
    # 7) 고령 여부 및 interaction
    # -----------------------------------------------------
    if "시술 당시 나이" in df.columns:
        old_age_values = [
            "만38-39세",
            "만40-42세",
            "만43-44세",
            "만45-50세",
        ]

        df["고령여부"] = (
            df["시술 당시 나이"].astype(str).isin(old_age_values)
        ).astype(int)

        if "수집된 신선 난자 수" in df.columns:
            df["고령_난자수_interaction"] = (
                df["고령여부"] *
                pd.to_numeric(df["수집된 신선 난자 수"], errors="coerce").fillna(0)
            )

        if "이식된 배아 수" in df.columns:
            df["고령_배아이식"] = (
                df["고령여부"] *
                pd.to_numeric(df["이식된 배아 수"], errors="coerce").fillna(0)
            )

        if "총 생성 배아 수" in df.columns:
            df["고령_배아생성"] = (
                df["고령여부"] *
                pd.to_numeric(df["총 생성 배아 수"], errors="coerce").fillna(0)
            )

        if "저장된 배아 수" in df.columns:
            df["고령_배아저장"] = (
                df["고령여부"] *
                pd.to_numeric(df["저장된 배아 수"], errors="coerce").fillna(0)
            )

        if "미세주입된 난자 수" in df.columns:
            df["고령_미세주입난자"] = (
                df["고령여부"] *
                pd.to_numeric(df["미세주입된 난자 수"], errors="coerce").fillna(0)
            )

    # -----------------------------------------------------
    # 8) 범주형 정리
    # -----------------------------------------------------
    categorical_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in categorical_cols:
        df[col] = df[col].astype(str)

    return df

In [ ]:
# =========================================================
# 3. Combo categorical feature
# =========================================================

def add_combo_columns(X):
    """
    Combo TE용 조합 category 생성.

    핵심:
    단일 범주형 변수보다
    나이 × 난자 출처,
    시술 유형 × 난자 출처
    같은 조합에서 성공률 패턴이 더 잘 드러날 수 있음.
    """
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = (
                X[col1].astype(str) + "_" + X[col2].astype(str)
            )
            combo_cols.append(new_col)

    return X, combo_cols

In [ ]:
# =========================================================
# 4. Leakage-free OOF Target Encoding
# =========================================================

def add_oof_target_encoding(
    X,
    y,
    cols,
    n_splits=5,
    smoothing=10,
    random_state=42
):
    """
    Leakage 방지 Target Encoding.

    train:
        각 fold의 validation 데이터는
        해당 fold의 train 데이터로 fit한 encoder로만 transform.
        즉 자기 자신의 target을 보지 않음.

    test:
        전체 train으로 fit한 final encoder로 transform.

    반환:
        X_te: OOF TE feature가 붙은 train
        final_encoder: test 변환용 encoder
    """
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder

In [ ]:
# =========================================================
# 5. Final feature set builder
# =========================================================

def build_feature_set(
    train_raw,
    test_raw,
    target_col="임신 성공 여부",
    n_splits=5,
    fold_seed=42,
    te_smoothing=10,
):
    """
    모델 학습 직전까지의 최종 feature set 생성.

    반환:
        X_stack       : train feature
        X_test_stack  : test feature
        y_stack       : target
        cat_cols      : categorical columns
        te_cols       : target encoding에 사용한 columns
    """
    # -----------------------------------------------------
    # 1) target / feature 분리
    # -----------------------------------------------------
    X_raw = train_raw.drop(columns=[target_col])
    y = train_raw[target_col].astype(int).reset_index(drop=True)

    X_test_raw = test_raw.copy()

    # ID 계열 제거
    id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

    X_raw = X_raw.drop(columns=id_cols, errors="ignore").reset_index(drop=True)
    X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore").reset_index(drop=True)

    # -----------------------------------------------------
    # 2) 기본 preprocessing + FE
    # -----------------------------------------------------
    X = data_preprocessing(X_raw)
    X_test = data_preprocessing(X_test_raw)

    # -----------------------------------------------------
    # 3) Combo columns
    # -----------------------------------------------------
    X_combo, combo_cols = add_combo_columns(X)
    X_test_combo, _ = add_combo_columns(X_test)

    # -----------------------------------------------------
    # 4) TE 대상 columns
    # -----------------------------------------------------
    base_te_cols = [
        "시술 시기 코드",
        "시술 유형",
        "특정 시술 유형",
        "배란 유도 유형",
        "난자 출처",
        "정자 출처",
        "배아 생성 주요 이유",
        "시술 당시 나이",
    ]

    base_te_cols = [
        col for col in base_te_cols
        if col in X_combo.columns
    ]

    te_cols = base_te_cols + combo_cols

    print("base_te_cols:", base_te_cols)
    print("combo_cols:", combo_cols)
    print("te_cols count:", len(te_cols))

    # -----------------------------------------------------
    # 5) OOF Target Encoding for train
    # -----------------------------------------------------
    X_te_combo, te_encoder = add_oof_target_encoding(
        X_combo,
        y,
        cols=te_cols,
        n_splits=n_splits,
        smoothing=te_smoothing,
        random_state=fold_seed,
    )

    # -----------------------------------------------------
    # 6) Target Encoding for test
    # -----------------------------------------------------
    test_te_values = te_encoder.transform(X_test_combo[te_cols])

    for col in te_cols:
        X_test_combo[f"{col}_TE"] = test_te_values[col].values

    # -----------------------------------------------------
    # 7) combo 원본 문자열 컬럼 제거
    #    combo는 TE 값만 사용
    # -----------------------------------------------------
    X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
    X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

    # -----------------------------------------------------
    # 8) train/test 컬럼 정렬
    # -----------------------------------------------------
    X_test_te_combo = X_test_te_combo[X_te_combo.columns]

    X_stack = X_te_combo.copy().reset_index(drop=True)
    X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
    y_stack = y.reset_index(drop=True)

    # -----------------------------------------------------
    # 9) categorical columns 정리
    # -----------------------------------------------------
    cat_cols = X_stack.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        X_stack[col] = X_stack[col].astype(str)
        X_test_stack[col] = X_test_stack[col].astype(str)

    # -----------------------------------------------------
    # 10) sanity check
    # -----------------------------------------------------
    assert list(X_stack.columns) == list(X_test_stack.columns)
    assert len(X_stack) == len(y_stack)
    assert len(X_test_stack) == len(test_raw)

    print("X_stack:", X_stack.shape)
    print("X_test_stack:", X_test_stack.shape)
    print("y_stack:", y_stack.shape)
    print("cat_cols:", len(cat_cols))

    return X_stack, X_test_stack, y_stack, cat_cols, te_cols

In [ ]:
# =========================================================
# Usage
# =========================================================

TRAIN_PATH = "data/train.csv"
TEST_PATH = "data/test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

X_stack, X_test_stack, y_stack, cat_cols, te_cols = build_feature_set(
    train_raw=train_raw,
    test_raw=test_raw,
    target_col="임신 성공 여부",
    n_splits=5,
    fold_seed=42,
    te_smoothing=10,
)

기본 FE는 난임 시술 프로세스를 따라 구성했다.

1. 경과일 결측 여부를 performed flag로 보존
2. 횟수형 문자열을 숫자화
3. 불임 원인을 남성/여성/복합/원인불명으로 요약
4. 배아 생성률, 이식률, 냉동률, 이식 집중도 같은 효율 feature 생성
5. 고령 여부와 난자/배아 수 interaction 생성
6. 주요 범주형 변수와 조합 범주형 변수에 대해 OOF Target Encoding 적용

Target Encoding은 반드시 fold 내부 train 데이터로만 fit해서 validation에 transform했다.
따라서 validation target이 encoding 과정에 새어 들어가는 leakage를 방지했다.